In [1]:
import os
import sys
import socket
import torch
import torchvision.transforms as transforms
from torchvision.models import resnet101, ResNet101_Weights
import numpy as np
from PIL import Image
import gradio as gr
import matplotlib.pyplot as plt
import pickle
import traceback
from datetime import datetime

# 设置matplotlib中文字体
plt.rcParams['font.sans-serif'] = ['Microsoft YaHei', 'SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

# 设置环境变量
os.environ['GRADIO_ANALYTICS_ENABLED'] = 'False'

def find_free_port(start_port=7860, max_attempts=10):
    """查找可用端口"""
    for port in range(start_port, start_port + max_attempts):
        try:
            with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
                s.bind(('127.0.0.1', port))
                return port
        except OSError:
            continue
    return None

class RobustImageRetrievalDemo:
    """稳定的图片检索演示系统"""
    
    def __init__(self, base_path, feature_path=None, device='cuda'):
        self.base_path = base_path
        self.device = device if torch.cuda.is_available() else 'cpu'
        print(f"[INFO] 使用设备: {self.device}")
        
        # 完整的建筑名缩写映射
        self.landmark_mapping = {
            'sy': '思源楼', 'tsg': '图书馆', 'nm': '南门', 'ty': '天佑会堂',
            'sjz': '世纪钟', 'zx': '知行碑', 'mh': '明湖碑', 'fhy': '芳华园碑',
            'yf': '逸夫楼', 'jx': '机械楼', 'kx': '科学会堂', 'yks': '迎客松',
            'kxht': '科学会堂', 'yk': '迎客松'
        }
        
        # 反向映射
        self.reverse_mapping = {v: k for k, v in self.landmark_mapping.items() if len(k) <= 3}
        
        # 数据预处理
        self.transform = transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])
        
        # 加载模型
        print("[INFO] 加载ResNet101模型...")
        try:
            self.model = resnet101(weights=ResNet101_Weights.IMAGENET1K_V1)
            self.model = torch.nn.Sequential(*list(self.model.children())[:-1])
            self.model = self.model.to(self.device)
            self.model.eval()
            print("[INFO] ResNet101模型加载完成")
        except Exception as e:
            print(f"[ERROR] 模型加载失败: {e}")
            traceback.print_exc()
            raise
        
        # 加载特征
        self.base_features = {}
        if feature_path and os.path.exists(feature_path):
            print(f"[INFO] 加载特征文件: {feature_path}")
            try:
                with open(feature_path, 'rb') as f:
                    self.base_features = pickle.load(f)
                print(f"[INFO] 成功加载 {len(self.base_features)} 张图片的特征")
            except Exception as e:
                print(f"[ERROR] 特征加载失败: {e}")
                traceback.print_exc()
        else:
            print(f"[WARNING] 特征文件不存在: {feature_path}")
            print("[INFO] 将尝试实时提取特征（速度较慢）")
        
        self.image_paths = list(self.base_features.keys())
        print(f"[INFO] 检索库准备就绪，共 {len(self.image_paths)} 张图片")
    
    def extract_features(self, image_path):
        """提取单张图片的特征"""
        try:
            if isinstance(image_path, str):
                image = Image.open(image_path).convert('RGB')
            else:
                image = image_path.convert('RGB')
            
            image_tensor = self.transform(image).unsqueeze(0).to(self.device)
            
            with torch.no_grad():
                features = self.model(image_tensor)
                features = features.squeeze().cpu().numpy()
            
            # L2归一化
            norm = np.linalg.norm(features)
            if norm > 0:
                features = features / norm
            
            return features
        except Exception as e:
            print(f"[ERROR] 特征提取失败: {e}")
            traceback.print_exc()
            return None
    
    def extract_landmark(self, filename):
        """从文件名提取建筑名称"""
        try:
            base_name = os.path.basename(filename)
            name_without_ext = os.path.splitext(base_name)[0]
            name_lower = name_without_ext.lower()
            
            # 优先检查科学会堂和迎客松
            if 'kxht' in name_lower or name_lower.startswith('kx'):
                return '科学会堂'
            if 'yks' in name_lower or name_lower.startswith('yk'):
                return '迎客松'
            
            # 按 '-' 分割
            if '-' in name_without_ext:
                code = name_without_ext.split('-')[0].lower()
                if code in self.landmark_mapping:
                    return self.landmark_mapping[code]
            
            # 匹配缩写
            for code, name in self.landmark_mapping.items():
                if name_lower.startswith(code):
                    return name
            
            # 从父文件夹获取
            parent = os.path.basename(os.path.dirname(filename))
            if parent in self.landmark_mapping:
                return self.landmark_mapping[parent]
            
            return "未知建筑"
        except Exception as e:
            print(f"[ERROR] 标签提取失败: {e}")
            return "未知建筑"
    
    def extract_landmark_code(self, filename):
        """从文件名提取建筑代码"""
        try:
            base_name = os.path.basename(filename)
            name_without_ext = os.path.splitext(base_name)[0]
            name_lower = name_without_ext.lower()
            
            if 'kxht' in name_lower or name_lower.startswith('kx'):
                return 'kx'
            if 'yks' in name_lower or name_lower.startswith('yk'):
                return 'yks'
            
            if '-' in name_without_ext:
                code = name_without_ext.split('-')[0].lower()
                if code in self.landmark_mapping:
                    return code
            
            for code in self.landmark_mapping.keys():
                if name_lower.startswith(code):
                    return code
            
            return None
        except Exception:
            return None
    
    def search(self, query_image, top_k=10):
        """检索相似图片"""
        print(f"[INFO] 开始检索，Top-K={top_k}")
        
        # 提取查询图片特征
        query_feat = self.extract_features(query_image)
        if query_feat is None:
            print("[ERROR] 查询图片特征提取失败")
            return [], [], [], "特征提取失败"
        
        query_code = self.extract_landmark_code(query_image)
        query_name = self.extract_landmark(query_image)
        print(f"[INFO] 查询图片识别为: {query_name} (code: {query_code})")
        
        # 如果没有预加载的特征，实时提取
        if not self.base_features:
            print("[WARNING] 没有预加载特征，将实时检索...")
            return self._search_real_time(query_feat, query_code, top_k)
        
        # 计算相似度
        similarities = []
        for idx, (img_path, base_feat) in enumerate(self.base_features.items()):
            try:
                sim = np.dot(query_feat, base_feat)
                base_code = self.extract_landmark_code(img_path)
                similarities.append((img_path, sim, base_code))
            except Exception as e:
                print(f"[WARNING] 计算相似度失败: {img_path}, {e}")
                continue
        
        # 排序
        similarities.sort(key=lambda x: x[1], reverse=True)
        top_results = similarities[:top_k]
        
        # 准备结果
        result_images = []
        result_names = []
        result_scores = []
        result_paths = []
        
        for img_path, score, base_code in top_results:
            full_path = os.path.join(self.base_path, img_path)
            try:
                if os.path.exists(full_path):
                    img = Image.open(full_path)
                    result_images.append(img)
                    name = self.landmark_mapping.get(base_code, "未知建筑") if base_code else "未知建筑"
                    result_names.append(name)
                    result_scores.append(score)
                    result_paths.append(img_path)
                else:
                    print(f"[WARNING] 图片不存在: {full_path}")
            except Exception as e:
                print(f"[WARNING] 加载图片失败: {full_path}, {e}")
        
        print(f"[INFO] 检索完成，找到 {len(result_images)} 张相似图片")
        return result_images, result_names, result_scores, result_paths
    
    def _search_real_time(self, query_feat, query_code, top_k=10):
        """实时检索（当没有预加载特征时使用）"""
        bjtu_path = os.path.join(self.base_path, 'BJTU')
        search_path = bjtu_path if os.path.exists(bjtu_path) else self.base_path
        
        similarities = []
        for root, dirs, files in os.walk(search_path):
            for file in files:
                if file.lower().endswith(('.png', '.jpg', '.jpeg')):
                    img_path = os.path.join(root, file)
                    try:
                        feat = self.extract_features(img_path)
                        if feat is not None:
                            sim = np.dot(query_feat, feat)
                            base_code = self.extract_landmark_code(img_path)
                            similarities.append((img_path, sim, base_code))
                    except Exception as e:
                        print(f"[WARNING] 处理失败: {img_path}, {e}")
        
        similarities.sort(key=lambda x: x[1], reverse=True)
        top_results = similarities[:top_k]
        
        result_images = []
        result_names = []
        result_scores = []
        
        for img_path, score, base_code in top_results:
            try:
                img = Image.open(img_path)
                result_images.append(img)
                name = self.landmark_mapping.get(base_code, "未知建筑") if base_code else "未知建筑"
                result_names.append(name)
                result_scores.append(score)
            except Exception as e:
                print(f"[WARNING] 加载图片失败: {img_path}, {e}")
        
        return result_images, result_names, result_scores, "实时检索完成"
    
    def visualize_results(self, query_image, results, result_names, result_scores, top_k=10):
        """可视化检索结果"""
        try:
            # 获取查询图片信息
            if isinstance(query_image, str):
                query_img = Image.open(query_image)
            else:
                query_img = query_image
            query_landmark = self.extract_landmark(query_image)
            
            # 创建图形
            fig = plt.figure(figsize=(20, 12))
            
            # 使用GridSpec布局
            gs = fig.add_gridspec(3, 5, hspace=0.3, wspace=0.3)
            
            # 查询图片区域
            ax_query = fig.add_subplot(gs[0:2, 0:2])
            ax_query.imshow(query_img)
            ax_query.set_title(f'查询图片\n地标: {query_landmark}', fontsize=14, fontweight='bold')
            ax_query.axis('off')
            
            # 显示检索结果
            display_count = min(top_k, len(results))
            for i in range(display_count):
                if i < 3:
                    row, col = 0, i + 2
                else:
                    row, col = 1, i - 3 + 2
                    if i >= 6:
                        row, col = 2, i - 6
                
                ax = fig.add_subplot(gs[row, col])
                ax.imshow(results[i])
                
                is_correct = (result_names[i] == query_landmark)
                title = f'排名 {i+1}\n相似度: {result_scores[i]:.3f}\n{result_names[i]}'
                if is_correct:
                    title += ' ✓'
                
                ax.set_title(title, fontsize=10)
                ax.axis('off')
                
                if is_correct:
                    for spine in ax.spines.values():
                        spine.set_edgecolor('green')
                        spine.set_linewidth(3)
            
            fig.suptitle('交大视觉印象 - 以图搜图检索结果', fontsize=16, fontweight='bold')
            plt.tight_layout()
            
            return fig
        except Exception as e:
            print(f"[ERROR] 可视化失败: {e}")
            traceback.print_exc()
            # 返回一个简单的错误提示图
            fig, ax = plt.subplots(figsize=(10, 5))
            ax.text(0.5, 0.5, f'可视化失败: {str(e)}', ha='center', va='center', fontsize=12)
            ax.axis('off')
            return fig


def create_demo():
    """创建Gradio演示界面"""
    
    # 配置路径
    base_path = r"E:\image_retrieval\base"
    feature_path = r"E:\image_retrieval\base_features_weighted.pkl"
    
    # 检查文件是否存在
    print(f"[INFO] 检查base路径: {base_path}, 存在: {os.path.exists(base_path)}")
    print(f"[INFO] 检查特征文件: {feature_path}, 存在: {os.path.exists(feature_path)}")
    
    # 初始化检索系统
    try:
        retrieval_sys = RobustImageRetrievalDemo(base_path, feature_path)
    except Exception as e:
        print(f"[ERROR] 初始化失败: {e}")
        traceback.print_exc()
        retrieval_sys = None
    
    def search_and_display(image, top_k):
        """Gradio接口函数"""
        if retrieval_sys is None:
            return None, "系统初始化失败，请检查日志"
        
        if image is None:
            return None, "请上传一张图片"
        
        print(f"\n{'='*50}")
        print(f"[INFO] 收到检索请求，top_k={top_k}")
        
        try:
            # 执行检索
            results, result_names, result_scores, debug_info = retrieval_sys.search(image, int(top_k))
            
            if not results:
                return None, f"未找到相似图片\n\n调试信息:\n{debug_info}"
            
            # 可视化结果
            fig = retrieval_sys.visualize_results(image, results, result_names, result_scores, int(top_k))
            
            # 文本结果
            text_result = "=" * 50 + "\n"
            text_result += f"检索结果 (Top-{len(results)}):\n"
            text_result += "=" * 50 + "\n\n"
            for i, (name, score) in enumerate(zip(result_names, result_scores)):
                text_result += f"{i+1}. {name} (相似度: {score:.4f})\n"
            
            # 添加调试信息
            text_result += f"\n{'='*50}\n"
            text_result += f"检索库图片数: {len(retrieval_sys.image_paths)}\n"
            text_result += f"查询图片地标: {retrieval_sys.extract_landmark(image)}\n"
            
            return fig, text_result
            
        except Exception as e:
            print(f"[ERROR] 检索过程出错: {e}")
            traceback.print_exc()
            return None, f"检索出错: {str(e)}"
    
    # 创建Gradio界面
    with gr.Blocks(title="交大视觉印象 - 以图搜图检索系统", theme=gr.themes.Soft()) as demo:
        gr.Markdown("""
        #  交大视觉印象 - 以图搜图检索系统
        
        <div style="background-color: #f0f8ff; padding: 10px; border-radius: 5px; margin-bottom: 15px;">
        <b> 系统信息</b><br>
        - 使用 ResNet101 提取图像特征<br>
        - 支持12个地标的检索：思源楼、图书馆、南门、天佑会堂、世纪钟、知行碑、明湖碑、芳华园碑、逸夫楼、机械楼、科学会堂、迎客松<br>
        - 对科学会堂等低准确率类别使用加权检索
        </div>
        """)
        
        with gr.Row():
            with gr.Column(scale=1):
                input_image = gr.Image(type="filepath", label="📤 上传查询图片")
                top_k_slider = gr.Slider(
                    minimum=5, maximum=30, value=10, step=5,
                    label="检索数量 (Top-K)"
                )
                search_btn = gr.Button(" 开始检索", variant="primary", size="lg")
                
                gr.Markdown("""
                ### 📷 测试建议
                1. 使用 query 文件夹中的图片进行测试
                2. 支持 JPG、PNG 格式
                3. 建议使用清晰、光照良好的图片
                """)
            
            with gr.Column(scale=2):
                output_image = gr.Plot(label=" 检索结果可视化")
                output_text = gr.Textbox(label=" 详细结果", lines=15)
        
        # 绑定事件
        search_btn.click(
            fn=search_and_display,
            inputs=[input_image, top_k_slider],
            outputs=[output_image, output_text]
        )
        
        # 添加示例（如果存在）
        query_path = r"E:\image_retrieval\query"
        if os.path.exists(query_path):
            examples = []
            for file in os.listdir(query_path)[:5]:
                if file.lower().endswith(('.jpg', '.png', '.jpeg')):
                    examples.append([os.path.join(query_path, file)])
            if examples:
                gr.Examples(
                    examples=examples,
                    inputs=[input_image],
                    label=" 示例图片（点击快速测试）"
                )
        
        gr.Markdown("""
        ---
        ###  说明
        - ✓ 标记表示检索结果与查询图片属于同一地标
        - 相似度范围: 0-1，数值越大表示越相似
        - 首次检索可能需要加载模型，请稍候
        """)
    
    return demo


def run_demo():
    """运行演示"""
    print("\n" + "="*60)
    print("启动交大视觉印象 - 以图搜图检索系统")
    print("="*60 + "\n")
    
    # 查找可用端口
    available_port = find_free_port(start_port=7860, max_attempts=10)
    if available_port is None:
        print("[WARNING] 未找到可用端口，将使用默认端口并尝试覆盖")
        available_port = 7860
    
    print(f"[INFO] 使用端口: {available_port}")
    
    try:
        demo = create_demo()
        
        # 启动服务
        demo.launch(
            share=False,
            server_name="127.0.0.1",
            server_port=available_port,
            debug=False,
            quiet=False
        )
    except OSError as e:
        print(f"[ERROR] 端口 {available_port} 无法使用: {e}")
        # 尝试更多端口
        for port in range(7861, 7880):
            try:
                print(f"[INFO] 尝试端口: {port}")
                demo.launch(
                    share=False,
                    server_name="127.0.0.1",
                    server_port=port,
                    debug=False,
                    quiet=True
                )
                break
            except OSError:
                continue
        else:
            print("[ERROR] 无法找到可用端口，请手动关闭占用端口的程序")
            print("可以使用以下命令查看占用端口的程序：")
            print("  netstat -ano | findstr :7860")
            print("  taskkill /PID <PID> /F")


if __name__ == "__main__":
    run_demo()

C:\Users\purpled\AppData\Roaming\Python\Python312\site-packages\pandas\core\computation\expressions.py:23: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.7' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED



启动交大视觉印象 - 以图搜图检索系统

[INFO] 使用端口: 7860
[INFO] 检查base路径: E:\image_retrieval\base, 存在: True
[INFO] 检查特征文件: E:\image_retrieval\base_features_weighted.pkl, 存在: True
[INFO] 使用设备: cuda
[INFO] 加载ResNet101模型...
[INFO] ResNet101模型加载完成
[INFO] 加载特征文件: E:\image_retrieval\base_features_weighted.pkl
[INFO] 成功加载 2662 张图片的特征
[INFO] 检索库准备就绪，共 2662 张图片


C:\Users\purpled\AppData\Local\Temp\ipykernel_25148\1366149117.py:392: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(title="交大视觉印象 - 以图搜图检索系统", theme=gr.themes.Soft()) as demo:


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.



[INFO] 收到检索请求，top_k=10
[INFO] 开始检索，Top-K=10
[INFO] 查询图片识别为: 未知建筑 (code: None)
[INFO] 检索完成，找到 10 张相似图片


...

C:\Users\purpled\AppData\Local\Temp\ipykernel_25148\1366149117.py:319: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()
d:\Anacoda\Lib\site-packages\gradio\processing_utils.py:82: UserWarning: Glyph 10003 (\N{CHECK MARK}) missing from font(s) Microsoft YaHei.
  plt.savefig(output_bytes, format=fmt)



[INFO] 收到检索请求，top_k=10
[INFO] 开始检索，Top-K=10
[INFO] 查询图片识别为: 芳华园碑 (code: fhy)
[INFO] 检索完成，找到 10 张相似图片


C:\Users\purpled\AppData\Local\Temp\ipykernel_25148\1366149117.py:319: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()
d:\Anacoda\Lib\site-packages\gradio\processing_utils.py:82: UserWarning: Glyph 10003 (\N{CHECK MARK}) missing from font(s) Microsoft YaHei.
  plt.savefig(output_bytes, format=fmt)



[INFO] 收到检索请求，top_k=10
[INFO] 开始检索，Top-K=10
[INFO] 查询图片识别为: 明湖碑 (code: mh)
[INFO] 检索完成，找到 10 张相似图片


C:\Users\purpled\AppData\Local\Temp\ipykernel_25148\1366149117.py:319: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()
d:\Anacoda\Lib\site-packages\gradio\processing_utils.py:82: UserWarning: Glyph 10003 (\N{CHECK MARK}) missing from font(s) Microsoft YaHei.
  plt.savefig(output_bytes, format=fmt)
